# Streaming

Streaming pozwala pokazywać fragmenty odpowiedzi w chwili, gdy przychodzą z modelu. Najpierw zbierzemy fragmenty ręcznie z `model.stream(...)`, potem z agenta przez `stream_mode="messages"`.

[Streaming w LangChain](https://docs.langchain.com/oss/python/langchain/streaming)


In [ ]:
# Imports and project path setup only; examples start in the next cell.
from functools import wraps
from pathlib import Path
from time import perf_counter
import json
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from langchain.agents import create_agent
from model_config import get_json_model, get_model
from pydantic import BaseModel, Field


In [1]:
model = get_model()


Using local Ollama model: gemma4:12b


### 1. Streaming: handle model chunks ourselves

`model.stream(...)` yields `AIMessageChunk` objects. We print each text piece immediately and join the pieces ourselves. A chunk is not necessarily one word or one token.

In [2]:
def measure_seconds(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        try:
            return function(*args, **kwargs)
        finally:
            elapsed = perf_counter() - start
            print(f"\n{function.__name__} took {elapsed:.2f} seconds")

    return wrapper

In [3]:
@measure_seconds
def stream_model_answer(question: str) -> str:
    pieces = []
    print("MODEL: ", end="", flush=True)

    for chunk in model.stream(question):
        text = str(chunk.text)
        print(text, end="", flush=True)
        pieces.append(text)

    answer = "".join(pieces)
    print(f"\nReceived {len(pieces)} chunks; assembled answer: {answer!r}")
    return answer

In [4]:
stream_model_answer("In an AI chat, what does streaming model output mean? Provide medium lenght and verbose answer please")

MODEL: Here are two explanations of what "streaming" means in the context of AI chat models, tailored to your requested lengths.

---

### Medium Length Answer

In AI chat, **streaming** refers to the method of delivering the model's response to the user piece-by-piece (usually word-by-word or character-by-character) as it is being generated, rather than waiting for the entire response to be completed before displaying it.

**How it works:**
Large Language Models (LLMs) do not "think" of a full paragraph instantly; they generate text one "token" (a chunk of a word or a whole word) at a time by predicting what comes next. In a **non-streaming** setup, the server waits until the entire paragraph is finished before sending the data to your screen. In a **streaming** setup, the server sends each token to your device the millisecond it is generated.

**Why it is used:**
1.  **Reduced Perceived Latency:** Even if the total time to generate a 500-word essay is the same, streaming makes the ex

'Here are two explanations of what "streaming" means in the context of AI chat models, tailored to your requested lengths.\n\n---\n\n### Medium Length Answer\n\nIn AI chat, **streaming** refers to the method of delivering the model\'s response to the user piece-by-piece (usually word-by-word or character-by-character) as it is being generated, rather than waiting for the entire response to be completed before displaying it.\n\n**How it works:**\nLarge Language Models (LLMs) do not "think" of a full paragraph instantly; they generate text one "token" (a chunk of a word or a whole word) at a time by predicting what comes next. In a **non-streaming** setup, the server waits until the entire paragraph is finished before sending the data to your screen. In a **streaming** setup, the server sends each token to your device the millisecond it is generated.\n\n**Why it is used:**\n1.  **Reduced Perceived Latency:** Even if the total time to generate a 500-word essay is the same, streaming makes

### 2. Streaming from a LangChain agent

`stream_mode="messages"` yields `(chunk, metadata)` pairs from the agent's model node. We display each text fragment as it arrives. In lesson 05, `stream_mode="updates"` showed completed model and tool steps instead.


In [ ]:
streaming_agent = create_agent(
    model=model,
    tools=[],
    system_prompt="Answer the question from user",
)


@measure_seconds
def stream_agent_answer(question: str) -> str:
    pieces = []
    print("AGENT: ", end="", flush=True)

    for chunk, metadata in streaming_agent.stream(
        {"messages": [{"role": "user", "content": question}]},
        stream_mode="messages",
    ):
        if metadata.get("langgraph_node") == "model":
            text = str(chunk.text)
            print(text, end="", flush=True)
            pieces.append(text)

    answer = "".join(pieces)
    print(f"\nReceived {len(pieces)} agent chunks; assembled answer: {answer!r}")
    return answer

In [ ]:
stream_agent_answer("In an AI chat, what does streaming model output mean? Provide medium lenght and verbose answer please")

AGENT: Structured output is the generation of data in a predefined format, such as JSON, to ensure consistency and ease of parsing.
Received 343 agent chunks; assembled answer: 'Structured output is the generation of data in a predefined format, such as JSON, to ensure consistency and ease of parsing.'

stream_agent_answer took 3.57 seconds


'Structured output is the generation of data in a predefined format, such as JSON, to ensure consistency and ease of parsing.'

### 3. JSON przychodzący fragmentami

Jeśli model streamuje JSON, pojedynczy fragment zwykle **nie jest poprawnym dokumentem JSON**. Zbieramy wszystkie kawałki, a `json.loads` i walidację Pydantic uruchamiamy dopiero po zakończeniu strumienia. Schematy i stan agenta zostały wyjaśnione w poprzedniej lekcji.


In [ ]:
class StreamSummary(BaseModel):
    topic: str = Field(min_length=1)
    explanation: str = Field(min_length=10)


json_model = get_json_model()
json_prompt = (
    "Wyjaśnij w jednym zdaniu, czym jest streaming odpowiedzi modelu. "
    "Zwróć wyłącznie obiekt JSON z polami topic i explanation."
)
pieces = []
print("Fragmenty JSON: ", end="", flush=True)
for chunk in json_model.stream(json_prompt):
    piece = str(chunk.text)
    print(piece, end="", flush=True)
    pieces.append(piece)

complete_json = "".join(pieces)
summary = StreamSummary.model_validate(json.loads(complete_json))
print("\nObiekt po zakończeniu strumienia:", summary)
